# Тестирование сервиса в ручную

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 500) 

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path("../.env").resolve()
load_dotenv(env_path)

HF_TOKEN = os.getenv("HF_TOKEN")

print(env_path)
print(HF_TOKEN is not None)

In [ ]:
import sys
import os

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_path)

from app.settings import settings

TEST_AUDIO_FILE = settings.AUDIO_DIR / "О проекте 5 мин.mp4"
CACHE_DIR = settings.CACHE_DIR

In [ ]:
import torch

def print_vram(prefix=""):
    if not torch.cuda.is_available():
        print("CUDA недоступна")
        return

    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    max_allocated = torch.cuda.max_memory_allocated() / 1024**3
    max_reserved = torch.cuda.max_memory_reserved() / 1024**3

    print(f"{prefix}")
    print(f"Allocated:     {allocated:.3f} GB")
    print(f"Reserved:      {reserved:.3f} GB")
    print(f"Max allocated: {max_allocated:.3f} GB")
    print(f"Max reserved:  {max_reserved:.3f} GB")

1. `ai-sage/GigaAM-v3`
   Русскоязычная ASR-модель с вариантами CTC и RNNT. Ориентирована прежде всего на русский язык.
   Преимущества: высокое качество русского, компактный размер, RNNT подходит для streaming, MIT-лицензия, сравнительно низкие требования к GPU.

2. `nvidia/parakeet-tdt-0.6b-v3`
   Многоязычная FastConformer + TDT модель NVIDIA на 600M параметров, поддерживает русский и другие европейские языки.
   Преимущества: высокая скорость, хорошая точность, многоязычность, пунктуация и таймкоды, развитая экосистема NeMo/Hugging Face.

3. `nvidia/nemotron-3.5-asr-streaming-0.6b`
   Специализированная многоязычная streaming-ASR модель NVIDIA на 600M параметров с cache-aware FastConformer + RNNT.
   Преимущества: полноценный streaming, русский поддерживается официально, настраиваемая задержка примерно от 80 мс до 1,1 с, хорошо подходит для большого числа live-потоков.

4. `nur-dev/realtime-streaming-asr-kk-ru-en`
   Компактная streaming-модель примерно на 115M параметров для русского, английского и казахского.
   Преимущества: настоящий cache-aware streaming, очень небольшой размер, низкие требования к GPU, минимальная latency. Главный недостаток — лицензия CC BY-NC 4.0, поэтому стандартную версию нельзя использовать коммерчески.

Для тестирования я бы шёл именно в таком порядке: **GigaAM-v3 → Nemotron 3.5 Streaming → Parakeet TDT → nur-dev**. Первые три наиболее интересны для production, а последняя — скорее как компактный технологический benchmark.

# GigaAM-v3

`revision` здесь выбирает конкретный вариант GigaAM-v3 внутри одного репозитория. Это не просто версия кода, а фактически разные ASR-модели с разными головами и назначением.

`ssl` — базовая self-supervised модель без готовой ASR-головы. Нужна в основном для fine-tuning и исследований. Для обычной транскрипции не подходит напрямую.

`ctc` — модель с CTC-декодером. Простая, быстрая и удобная для обычной транскрипции. Как правило, легче и быстрее RNNT, но хуже приспособлена к настоящему streaming.

`rnnt` — модель с RNN-T декодером. Основной вариант для потокового распознавания речи. RNN-T естественно работает с последовательным поступлением аудио и поэтому лучше подходит для real-time ASR.

`e2e_ctc` — end-to-end CTC. Помимо распознавания речи, модель сразу стремится выдавать более готовый текст: с нормализацией, пунктуацией и оформлением. Хороший вариант для offline-транскрипции.

`e2e_rnnt` — end-to-end RNN-T. Сочетает преимущества RNNT для streaming с более готовым оформлением текста. Для вашей задачи это наиболее интересный вариант.

Упрощённо:

| Revision   | Распознавание |   Streaming | Готовый текст | Основное применение           |
| ---------- | ------------: | ----------: | ------------: | ----------------------------- |
| `ssl`      |           Нет |           — |           Нет | Fine-tuning                   |
| `ctc`      |            Да | Ограниченно |       Базовый | Быстрый offline ASR           |
| `rnnt`     |            Да |      **Да** |       Базовый | **Real-time ASR**             |
| `e2e_ctc`  |            Да | Ограниченно |        **Да** | Качественный offline ASR      |
| `e2e_rnnt` |            Да |      **Да** |        **Да** | **Real-time + готовый текст** |


In [ ]:
MODEL_NAME = "ai-sage/GigaAM-v3"
MODEL_REVISION = "e2e_rnnt"

In [ ]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB",
    )

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    cache_dir=CACHE_DIR,
    token=HF_TOKEN,
)

model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

In [ ]:

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

before = torch.cuda.memory_allocated()


In [ ]:
transcription = model.transcribe_longform(str(TEST_AUDIO_FILE))

torch.cuda.synchronize()

print(transcription)

In [ ]:
after = torch.cuda.memory_allocated()
peak = torch.cuda.max_memory_allocated()

print(f"До:        {before / 1024**3:.3f} GB")
print(f"После:     {after / 1024**3:.3f} GB")
print(f"Пик:       {peak / 1024**3:.3f} GB")
print(f"Рост пик:  {(peak - before) / 1024**3:.3f} GB")

In [ ]:
import subprocess

STREAM_AUDIO_FILE = CACHE_DIR / "gigaam_stream_test.wav"

subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i", str(TEST_AUDIO_FILE),
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        "-c:a", "pcm_s16le",
        str(STREAM_AUDIO_FILE),
    ],
    check=True,
)

print(STREAM_AUDIO_FILE)

In [ ]:
import torchaudio

waveform, sample_rate = torchaudio.load(STREAM_AUDIO_FILE)

print("Shape:", waveform.shape)
print("Sample rate:", sample_rate)
print("Duration:", waveform.shape[1] / sample_rate, "sec")

In [ ]:
import os
import time
import tempfile
import torch
import torchaudio
import pandas as pd

CHUNK_SEC = 0.5
WINDOW_SEC = 5.0

chunk_samples = int(CHUNK_SEC * sample_rate)
window_samples = int(WINDOW_SEC * sample_rate)

results = []

for end_sample in range(chunk_samples, waveform.shape[1] + 1, chunk_samples):
    start_sample = max(0, end_sample - window_samples)

    audio_window = waveform[:, start_sample:end_sample]

    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp_path = tmp.name

    torchaudio.save(
        tmp_path,
        audio_window,
        sample_rate,
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    text = model.transcribe(tmp_path)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    inference_time = time.perf_counter() - t0

    os.remove(tmp_path)

    audio_time = end_sample / sample_rate

    results.append({
        "audio_time": audio_time,
        "window_start": start_sample / sample_rate,
        "window_end": audio_time,
        "inference_time": inference_time,
        "text": text,
    })

    print(
        f"[{audio_time:6.1f}s] "
        f"inference={inference_time:.3f}s | "
        f"{text}"
    )

In [ ]:
df_stream = pd.DataFrame(results)

display(df_stream)

print("Mean latency:", df_stream["inference_time"].mean())
print("P95 latency:", df_stream["inference_time"].quantile(0.95))
print("Max latency:", df_stream["inference_time"].max())

df_stream["realtime_ok"] = (
    df_stream["inference_time"] < CHUNK_SEC
)

print(
    "Realtime chunks:",
    f'{df_stream["realtime_ok"].mean() * 100:.1f}%'
)